# 9 · Case study — environmental impact reports (*Miljørapport*)

A real-world process modeled in Conductor: the automated *miljørapport* pipeline
that the Danish Planning and Rural Development Agency (PLST) uses to generate
environmental impact reports for energy parks. The original system is ~50 Python
modules glued together with `asyncio.create_task` + `ThreadPoolExecutor`. This
notebook captures the same shape with Conductor primitives — node registration,
edges, **shared references**, **for-each regions**, **decision nodes with edge
guards**, and a **human-in-the-loop pause**.

The pipeline does:

1. **Parse** the user's *afgrænsningsnotat* (PDF/DOCX scoping document).
2. **Extract** project context with an LLM (cached on the document hash).
3. **Fetch** external data from EA-Tools / Plandata / Miljøstyrelsen — parallel
   fan-out over `LookupSpec`s, deduplicated across chapters.
4. **Generate** 14 chapters: 3 deterministic templates + 10 LLM chapters + 1
   cumulative chapter that depends on all the others. Resume runs after.
5. **Evaluate + revise** each LLM chapter — structural checks + LLM judges
   feed a decision; failures route to a revision call.
6. **Ground** each chapter (claim → source markdown table).
7. **Assemble** the final docx with map and source appendices.

We model this with stubbed services so the notebook runs with no API keys.
The point is the *shape* — which Conductor primitive maps to which stage.


## 1 · Stub services

A handful of fakes stand in for the real LLM, EA-Tools and PDF clients so the
notebook runs offline. They're deterministic — same input, same output —
which makes the example reproducible. The fakes return realistic Danish prose
so the drill-down sections lower in the notebook show what each node actually
produced.


In [ ]:
from __future__ import annotations

import textwrap
import time
from typing import Annotated, Any


# Fake LLM. The real one is GPT-4 with chapter-specific system prompts; here we
# pattern-match on prompt keywords and return realistic Danish prose so the
# drill-down later shows recognizable miljørapport content.
def fake_llm(prompt: str, *, model: str = "gpt-stub") -> str:
    p = prompt.lower()

    # Prompt-type matches go first — chapter-content matches below would
    # otherwise pick up chapter titles that appear inside summary prompts.
    if "summarize this project" in p or "summariser" in p:
        return (
            "Projektet omfatter etablering af en 120 MW solcellepark ved Borris i "
            "Vest- og Midtjylland. Det samlede projektareal er 240 ha agerland, "
            "hvoraf ca. 195 ha bliver til opstilling af paneler og driftsveje. "
            "Nærmeste Natura 2000-område (Borris Hede) ligger 6,2 km mod nordvest. "
            "Projektet kræver et kommuneplantillæg samt VVM-redegørelse efter "
            "miljøvurderingsloven, jf. § 8, stk. 1, nr. 1."
        )
    if "plain-language summary" in p or "borgervenlig" in p or "summary of:" in p:
        return (
            "## Resume\n\n"
            "Solcelleparken ved Borris er på 120 MW og vil dække ca. 240 "
            "hektar agerland. Den vil producere strøm svarende til ca. "
            "32 000 husstandes årlige forbrug. Vurderingen viser moderate "
            "påvirkninger på landskab og biologisk mangfoldighed, og "
            "ubetydelige påvirkninger på jord, vand og kulturmiljø. "
            "Med de foreslåede afværgende tiltag (afskærmende beplantning, "
            "anlægsstop i yngletiden 1. april–31. august, bufferzoner ved "
            "§ 3-områder) vurderes projektet at kunne realiseres uden "
            "væsentlige miljøpåvirkninger."
        )
    if "fix these issues" in p:
        return (
            "Det manglende juridiske grundlag er nu indsat: vurderingen "
            "udarbejdes efter miljøvurderingsloven, jf. § 20, stk. 2, samt "
            "naturbeskyttelseslovens § 3. Afsnittet er udvidet med relevante "
            "afværgende tiltag og en kort konklusion om "
            "påvirkningsgraden."
        )

    # Per-chapter generation. Match on the chapter title in the prompt.
    if "4. landskab" in p or "kapitel 4" in p:
        return (
            "Området har karakter af åbent agerland uden væsentlige "
            "landskabselementer. Terrænet er let kuperet (kote 38–46 m DVR90) "
            "og afgrænses mod nord af et levende hegn samt en mindre lavning. "
            "Den planlagte solcellepark vurderes at give en moderat visuel "
            "påvirkning fra Højmarkvej (afstand 480 m) og fra de tre nærmeste "
            "ejendomme (afstand 250–600 m). Med supplerende afskærmende "
            "beplantning vurderes påvirkningen at være mindre."
        )
    if "6. jord" in p or "kapitel 6" in p:
        return (
            "Jordbundsforholdene er overvejende sandblandet lerjord (JB4) med "
            "spredte tørveholdige indslag i lavningen mod nord. Området er ikke "
            "kortlagt som forurenet jord (V1/V2) i Region Midtjyllands database. "
            "Anlægsfasen kræver let afgravning til fundamenter og kabeltracéer; "
            "samlet jordhåndtering anslås til ca. 4 200 m³, som genanvendes på "
            "stedet i overensstemmelse med jordforureningslovens § 50, stk. 1."
        )
    if "9. vand" in p or "kapitel 9" in p:
        return (
            "Området afvandes til Borris Bæk via et eksisterende drænsystem. "
            "Der er registreret to mindre vandhuller (§ 3-beskyttede søer) "
            "henholdsvis 142 m og 310 m fra projektgrænsen. Begge friholdes "
            "fra anlægsarbejder med en bufferzone på minimum 10 m. "
            "Grundvandsstanden ligger 2,8–3,4 m u.t. og påvirkes ikke af "
            "anlægget. Risikoen for udvaskning af forurenende stoffer "
            "vurderes som ubetydelig, da paneler og fundamenter er passive."
        )
    if "10. biologisk" in p or "kapitel 10" in p:
        return (
            "Inden for 10 km radius er der registreret 1 Natura 2000-område "
            "(N225 Borris Hede), 4 § 3-naturtyper og habitat for to "
            "bilag IV-arter (markfirben, stor vandsalamander). En "
            "habitatscreening efter habitatbekendtgørelsens § 6, stk. 1 "
            "konkluderer, at projektet ikke i sig selv eller i forbindelse "
            "med andre planer vil skade udpegningsgrundlaget for "
            "N225. Markfirben-bestanden friholdes ved at undgå anlægsarbejde "
            "i perioden 1. april – 31. august."
        )

    # Cumulative chapter (ch14).
    if "14. kumulative" in p or "cumulative" in p:
        return (
            "De kumulative effekter af projektet er vurderet i forhold til "
            "den eksisterende vindmøllepark Borris Vest (8 møller, 4,2 km Ø) "
            "samt det igangværende kommuneplantillæg for et 80 MW "
            "solcelleanlæg i Borris Sø-området. Samlet vurderes den "
            "kumulative påvirkning på landskab, fugleliv og afvandingsforhold "
            "at være moderat, men under tærsklerne i miljøvurderingslovens "
            "bilag 1 nr. 3."
        )

    # Generic fallback — never hit when prompts are well-formed.
    head = prompt.splitlines()[0][:80] if prompt else "(empty)"
    return f"[{model}] generisk afsnit (uddybes i produktion). Prompt: {head}…"


# Fake EA-Tools / EA ESDH lookup. Returns realistic per-template excerpts so
# the drill-down can show what each chapter actually consumed.
def fake_ea_lookup(spec: "LookupSpec") -> str:
    if spec.template_id == "landskab":
        return (
            "=== EA-rapport (landskab, buffer 1 000 m) ===\n"
            "- Beskyttelseslinjer: 1 å-beskyttelseslinje (Borris Bæk) i kanten "
            "af bufferen\n"
            "- Større landskabselementer: ingen fredede arealer eller "
            "landskabsfredninger\n"
            "- Visuelle modtagere: 3 ejendomme inden for 600 m, 1 offentlig "
            "vej (Højmarkvej, 480 m)"
        )
    if spec.template_id == "jord":
        return (
            "=== EA-rapport (jord, buffer 100 m) ===\n"
            "- Jordart (JB-klassifikation): JB4 sandblandet lerjord, "
            "spredte tørveindslag\n"
            "- Forureningskortlægning: ingen V1/V2 (Region Midtjylland)\n"
            "- Råstofinteresser: området er ikke udpeget som råstofområde "
            "i råstofplanen 2024–2027"
        )
    if spec.template_id == "vand":
        return (
            "=== EA-rapport (vand, buffer 1 000 m) ===\n"
            "- § 3-vandhuller: 2 stk. (142 m N, 310 m NØ)\n"
            "- Vandløb: Borris Bæk (B-målsætning), passerer 380 m S\n"
            "- Drikkevandsinteresse: OD (område med drikkevandsinteresse), "
            "ikke OSD\n"
            "- Grundvandsspejl: 2,8–3,4 m u.t. (DGU 98.123)"
        )
    if spec.template_id == "natura2000":
        return (
            "=== EA-rapport (natura2000, buffer 10 000 m) ===\n"
            "- Natura 2000: N225 Borris Hede (6,2 km NV)\n"
            "- § 3-naturtyper: 4 (heraf 2 søer, 1 mose, 1 overdrev)\n"
            "- Bilag IV-arter: markfirben, stor vandsalamander (kendte "
            "lokaliteter inden for 5 km)"
        )
    return f"=== EA-rapport ({spec.template_id}, buffer {spec.buffer_m} m) ===\nIngen relevante data."


# Fake PDF/DOCX parser. The real service runs Tesseract on scanned
# afgrænsningsnotater; we return a deterministic dansk text so the rest
# of the flow has something realistic to read.
def fake_parse_document(blob: bytes) -> str:
    return textwrap.dedent(
        '''\
        AFGRÆNSNINGSNOTAT — Solcellepark Borris Energypark

        Bygherre: Borris Energy ApS, CVR 39 12 45 67.
        Projektet omfatter en 120 MW solcellepark på 240 ha agerland
        ved Borris, Ringkøbing-Skjern Kommune. Projektarealet er
        omfattet af kommuneplanramme 17.E.05.

        Afstand til nærmeste Natura 2000-område (Borris Hede): 6,2 km.
        Afstand til nærmeste § 3-vandhul: 142 m.
        Afgrænsning: kapitel 1–14 efter miljøvurderingslovens bilag 7.
        Forventet anlægsperiode: 2026 Q3 – 2027 Q1.
        '''
    )


## 2 · Domain types

Mirroring the envir backend's `chapter_types.py` and `lookup_data_config.py`,
trimmed to the bits the example exercises.


In [ ]:
from pydantic import BaseModel


class LookupSpec(BaseModel):
    template_id: str
    buffer_m: int
    label: str


class ChapterDraft(BaseModel):
    chapter_id: str
    title: str
    markdown: str
    sections: dict[str, str] = {}


class EvalResult(BaseModel):
    chapter_id: str
    score: float            # 0.0 (broken) … 1.0 (clean pass)
    issues: list[str] = []


# Trimmed chapter list — full system has 14.
TEMPLATE_CHAPTERS = [
    ("ch1", "1. Indledning"),
    ("ch2", "2. Bekendtgørelsens indhold"),
    ("ch3", "3. Afgrænsning"),
]
LLM_CHAPTERS = [
    ("ch4", "4. Landskab"),
    ("ch6", "6. Jord"),
    ("ch9", "9. Vand"),
    ("ch10", "10. Biologisk mangfoldighed"),
]
CUMULATIVE_CHAPTER = ("ch14", "14. Kumulative effekter")

# Static per-chapter EA spec config (envir's CHAPTER_LOOKUPS, condensed).
LOOKUP_CONFIG: dict[str, list[LookupSpec]] = {
    "ch4":  [LookupSpec(template_id="landskab",   buffer_m=1_000,  label="Landskab og beskyttelseslinjer")],
    "ch6":  [LookupSpec(template_id="jord",       buffer_m=100,    label="Jordforhold")],
    "ch9":  [LookupSpec(template_id="vand",       buffer_m=1_000,  label="Vandløb og søer")],
    "ch10": [LookupSpec(template_id="natura2000", buffer_m=10_000, label="Arter og beskyttede naturtyper")],
}


## 3 · Register the conductor nodes

Each node wraps one stage. Function-based registration; widget annotations
double as input metadata for any frontend that reads the registry.

Each chapter draft is built from four named **sections** (`legal_basis`,
`existing_conditions`, `assessment`, `conclusion`) — that mirrors the
envir templates and lets the drill-down pull each piece out separately.


In [ ]:
import conductor_nodes
from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.compound.for_each import FOR_EACH
from conductor.errors import FlowPausedException, HumanInputRequired
from conductor.execution.engine import collect, execute, resume
from conductor.widgets import Output, Text, Textarea

registry = NodeRegistry()

# Standard for-each markers (used in the parallel fan-out section).
conductor_nodes.loop.register(registry)


@registry.node("parse-scoping", version=1,
               name="Parse scoping note",
               description="PDF/DOCX → plain text")
def parse_scoping(
    blob_id: Annotated[str, Text(label="Document id")],
) -> Annotated[str, Output(label="Text")]:
    # Stand-in for PDF/DOCX extraction. In production the bytes come from
    # an upload; we look up by id for notebook reproducibility.
    return fake_parse_document(blob_id.encode())


@registry.node("extract-context", version=1,
               name="Extract project context",
               description="LLM extracts a one-paragraph project summary",
               idempotency_key='"ctx-" + string(size(scoping_text))')
def extract_context(
    scoping_text: Annotated[str, Textarea(label="Scoping text")],
    idempotency_key: str | None = None,
) -> Annotated[str, Output(label="Context")]:
    # idempotency_key (CEL-evaluated) survives retries — wired but unused here.
    return fake_llm(f"Summarize this project:\n{scoping_text}", model="gpt-light")


# Deduplicated EA specs as plain dicts — used as static data on
# `for-each-start.items` so the engine knows the iteration count up front.
def unique_lookup_specs() -> list[dict[str, Any]]:
    seen: set[tuple[str, int]] = set()
    out: list[dict[str, Any]] = []
    for specs in LOOKUP_CONFIG.values():
        for s in specs:
            key = (s.template_id, s.buffer_m)
            if key not in seen:
                seen.add(key)
                out.append(s.model_dump())
    return out


@registry.node("fetch-ea", version=1,
               name="Fetch EA spec",
               description="One EA-Tools call (parallel inside for-each)",
               max_retries=2, retry_delay=0.1)
def fetch_ea(
    spec: Annotated[LookupSpec, Text(label="Spec")],
) -> Annotated[dict[str, str], Output(label="Result")]:
    # Conductor stores all values as plain dicts on the wire; pydantic
    # validates on the way in but the function receives the dict form.
    # Reconstitute the model for dot access:
    spec = LookupSpec.model_validate(spec)
    # Real client would submit + poll + download. We sleep briefly
    # so parallelism is visible in node_start/node_complete timing.
    time.sleep(0.05)
    return {"key": f"{spec.template_id}@{spec.buffer_m}", "text": fake_ea_lookup(spec)}


@registry.node("index-results", version=1,
               name="Index EA results",
               description="list[ea_result] → dict[spec_key → text]")
def index_results(
    results: Annotated[list[dict[str, str]], Text(label="Results")],
) -> Annotated[dict[str, str], Output(label="Index")]:
    return {r["key"]: r["text"] for r in results}


@registry.node("template-chapter", version=1,
               name="Template chapter",
               description="Deterministic template fill (chapters 1–3)")
def template_chapter(
    chapter_id: Annotated[str, Text(label="Chapter id")],
    title: Annotated[str, Text(label="Title")],
    scoping_text: Annotated[str, Textarea(label="Scoping text")],
) -> Annotated[ChapterDraft, Output(label="Draft")]:
    # Chapters 1–3 are deterministic templates: introduction, regulatory
    # basis (miljøvurderingsloven), and scoping reference. We render
    # chapter-specific Danish prose so the drill-down has something to show.
    first_line = scoping_text.splitlines()[0] if scoping_text else "(ingen scoping-tekst)"
    if chapter_id == "ch1":
        body = (
            f"Denne miljørapport er udarbejdet for {first_line}. "
            "Rapporten følger strukturen i miljøvurderingslovens bilag 7 og "
            "behandler de miljøtemaer, der er afgrænset i samråd med "
            "miljømyndigheden."
        )
    elif chapter_id == "ch2":
        body = (
            "Miljøvurderingen udarbejdes i medfør af miljøvurderingslovens "
            "§ 20, stk. 2 (LBK nr. 4 af 03/01/2023). Projektet er omfattet "
            "af bilag 1, nr. 3 (anlæg til energiproduktion) og kræver "
            "VVM-redegørelse samt kommuneplantillæg."
        )
    else:  # ch3
        body = (
            "Afgrænsningen er gennemført på baggrund af et afgrænsningsnotat "
            "samt en høring af berørte myndigheder. Følgende temaer er "
            "afgrænset til vurdering: landskab, jord, vand, biologisk "
            "mangfoldighed, klima og kumulative effekter."
        )
    md_text = f"## {title}\n\n{body}"
    return ChapterDraft(chapter_id=chapter_id, title=title, markdown=md_text,
                        sections={"body": body})


@registry.node("llm-chapter", version=1,
               name="LLM chapter",
               description="Heavy LLM call (chapters 4–13)",
               max_retries=1, retry_delay=0.1)
def llm_chapter(
    chapter_id: Annotated[str, Text(label="Chapter id")],
    title: Annotated[str, Text(label="Title")],
    project_context: Annotated[str, Textarea(label="Project context")],
    ea_index: Annotated[dict[str, str], Text(label="EA index")],
) -> Annotated[ChapterDraft, Output(label="Draft")]:
    spec_keys = [f"{s.template_id}@{s.buffer_m}" for s in LOOKUP_CONFIG.get(chapter_id, [])]
    ea_text = "\n".join(ea_index.get(k, "(ingen data)") for k in spec_keys)

    # Each section is its own LLM call in production. Here we synthesize
    # them directly (legal_basis, assessment, conclusion are short formulaic
    # paragraphs) and call fake_llm only for `existing_conditions`, the
    # heavy chapter-body section that consumes the EA-Tools data.
    chapter_topic = title.lower().split(". ", 1)[-1]
    legal_basis = (
        f"Vurderingen af **{chapter_topic}** udarbejdes efter "
        "miljøvurderingslovens § 20, stk. 2, sammenholdt med "
        "naturbeskyttelseslovens § 3 (for arealtyper) og "
        "habitatbekendtgørelsens § 6 (for Natura 2000)."
    )
    existing = fake_llm(
        f"Generate chapter {title}.\nContext:\n{project_context}\nData:\n{ea_text}",
        model="gpt-heavy",
    )
    assessment_lookup = {
        "ch4":  "Den visuelle påvirkning vurderes som **moderat**. Med supplerende afskærmende beplantning langs Højmarkvej og mod de tre nærmeste ejendomme reduceres påvirkningen til mindre.",
        "ch6":  "Påvirkningen på jordbundsforholdene vurderes som **mindre**. Anlægsarbejder begrænses til let afgravning, og al jord genanvendes på stedet.",
        "ch9":  "Påvirkningen på vandløb og grundvand vurderes som **ubetydelig**. § 3-vandhuller friholdes med 10 m bufferzoner; der etableres olieudskillere ved transformerstationer.",
        "ch10": "Påvirkningen på biologisk mangfoldighed vurderes som **moderat**. Markfirben-bestanden friholdes ved at undgå anlægsarbejde i perioden 1. april – 31. august; der etableres kompensationshabitat på 4 ha.",
    }
    assessment = assessment_lookup.get(
        chapter_id,
        f"Påvirkningen for {chapter_topic} vurderes som **moderat**. "
        "Med de foreslåede afværgende tiltag forventes ingen væsentlige "
        "resterende påvirkninger.",
    )
    conclusion_lookup = {
        "ch4":  "**Konklusion:** Den samlede påvirkning på landskab og beskyttelseslinjer vurderes at være under tærsklen for væsentlig miljøpåvirkning.",
        "ch6":  "**Konklusion:** Projektet vurderes ikke at give væsentlig påvirkning på jord eller råstoffer.",
        "ch9":  "**Konklusion:** Projektet vurderes ikke at give væsentlig påvirkning på vandmiljøet.",
        "ch10": "**Konklusion:** Projektet vurderes ikke at skade udpegningsgrundlaget for N225 Borris Hede eller bilag IV-arter.",
    }
    conclusion = conclusion_lookup.get(
        chapter_id,
        f"**Konklusion:** Med de beskrevne afværgende tiltag vurderes "
        f"projektets påvirkning på {chapter_topic} at være under tærsklen "
        "for væsentlig miljøpåvirkning.",
    )

    sections = {
        "legal_basis":         legal_basis,
        "existing_conditions": existing,
        "assessment":          assessment,
        "conclusion":          conclusion,
    }
    md_text = (
        f"## {title}\n\n"
        f"### Lovgrundlag\n{legal_basis}\n\n"
        f"### Eksisterende forhold\n{existing}\n\n"
        f"### Vurdering\n{assessment}\n\n"
        f"### Konklusion\n{conclusion}\n"
    )
    return ChapterDraft(chapter_id=chapter_id, title=title, markdown=md_text,
                        sections=sections)


@registry.node("evaluate", version=1,
               name="Evaluate chapter",
               description="Structural + judge LLMs → score")
def evaluate(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
) -> Annotated[EvalResult, Output(label="Eval")]:
    draft = ChapterDraft.model_validate(draft)
    issues: list[str] = []
    # Real evaluator runs structural checks + 7 parallel LLM judges; we
    # model a few structural rules so the score is meaningful.
    if "legal_basis" not in draft.sections:
        issues.append("mangler afsnit om lovgrundlag")
    if "conclusion" not in draft.sections:
        issues.append("mangler konklusion")
    if len(draft.markdown) < 200:
        issues.append("for kort — under 200 tegn")
    if "natura 2000" in draft.title.lower() and "habitat" not in draft.markdown.lower():
        issues.append("kapitel om biologi mangler habitatvurdering")
    score = 1.0 if not issues else max(0.0, 1.0 - 0.3 * len(issues))
    return EvalResult(chapter_id=draft.chapter_id, score=score, issues=issues)


from conductor._sentinel import SKIPPED


# Router node using the SKIPPED-sentinel pattern. The decision-node /
# edge-guard feature is great for 1:1 routing (gate one outgoing edge by
# CEL), but here we need to route TWO values together — `draft` and
# `eval_result` should both reach `rev`, or neither. The SKIPPED sentinel
# composes naturally: each output handle is either a real value or
# SKIPPED, and downstream nodes whose inputs are all SKIPPED are skipped.
@registry.node("route-by-score", version=1,
               name="Route by score",
               description="Splits the chapter into revise/pass-through branches")
def route_by_score(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
    eval_result: Annotated[EvalResult, Text(label="Eval")],
) -> tuple[
    Annotated[Any, Output(label="Draft → revise")],
    Annotated[Any, Output(label="Eval → revise")],
    Annotated[Any, Output(label="Draft → pass-through")],
]:
    eval_result = EvalResult.model_validate(eval_result)
    if eval_result.score < 0.7:
        return draft, eval_result.model_dump(), SKIPPED
    return SKIPPED, SKIPPED, draft


@registry.node("revise", version=1,
               name="Revise chapter",
               description="LLM revision call",
               max_retries=2, retry_delay=0.1)
def revise(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
    eval_result: Annotated[EvalResult, Text(label="Eval")],
) -> Annotated[ChapterDraft, Output(label="Revised")]:
    draft = ChapterDraft.model_validate(draft)
    eval_result = EvalResult.model_validate(eval_result)
    feedback = "; ".join(eval_result.issues) or "(ingen)"
    prompt = f"Fix these issues in {draft.title}: {feedback}\n\n{draft.markdown}"
    revised_body = fake_llm(prompt, model="gpt-heavy")
    sections = {**draft.sections, "existing_conditions": revised_body}
    md_text = (
        f"## {draft.title} (revideret)\n\n"
        + "\n\n".join(sections.values())
    )
    return ChapterDraft(chapter_id=draft.chapter_id, title=draft.title,
                        markdown=md_text, sections=sections)


@registry.node("ground", version=1,
               name="Ground chapter",
               description="Claim → source table")
def ground(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
) -> Annotated[ChapterDraft, Output(label="Grounded")]:
    draft = ChapterDraft.model_validate(draft)
    # In production this is a separate LLM judge that extracts each numeric
    # claim from the draft and links it to a source layer in the EA index.
    table = (
        "\n\n### Kilder\n"
        "| Påstand | Kilde |\n"
        "|---|---|\n"
        f"| Afstand til Natura 2000 | EA-rapport (natura2000@10000) |\n"
        f"| § 3-vandhuller | EA-rapport (vand@1000) |\n"
        f"| Jordart JB4 | EA-rapport (jord@100) |\n"
    )
    return ChapterDraft(
        chapter_id=draft.chapter_id, title=draft.title,
        markdown=draft.markdown + table,
        sections={**draft.sections, "sources": table},
    )


@registry.node("cumulative", version=1,
               name="Cumulative chapter",
               description="Chapter 14 — depends on all earlier chapters")
def cumulative(
    title: Annotated[str, Text(label="Title")],
    earlier_chapters: Annotated[list[ChapterDraft], Text(label="Earlier chapters")],
) -> Annotated[ChapterDraft, Output(label="Draft")]:
    earlier_chapters = [ChapterDraft.model_validate(c) for c in earlier_chapters]
    bullets = "\n".join(
        f"- **{c.title}**: påvirkning på regionalt niveau vurderes som mindre."
        for c in earlier_chapters
    )
    body = fake_llm(f"Cumulative chapter 14. kumulative effekter\n{bullets}", model="gpt-heavy")
    md_text = f"## {title}\n\n{body}\n\n### Bidragende kapitler\n{bullets}"
    return ChapterDraft(chapter_id="ch14", title=title, markdown=md_text,
                        sections={"body": body, "contributors": bullets})


@registry.node("resume", version=1,
               name="Resume",
               description="Citizen-friendly summary across all chapters")
def resume_node(
    chapters: Annotated[list[ChapterDraft], Text(label="All chapters")],
) -> Annotated[str, Output(label="Resume markdown")]:
    chapters = [ChapterDraft.model_validate(c) for c in chapters]
    titles = ", ".join(c.title for c in chapters)
    return fake_llm(f"Plain-language summary of: {titles}", model="gpt-light")


@registry.node("assemble-docx", version=1,
               name="Assemble docx",
               description="Stitches chapters + appendices into final artifact")
def assemble_docx(
    chapters: Annotated[list[ChapterDraft], Text(label="Chapters")],
    resume_text: Annotated[str, Textarea(label="Resume")],
) -> Annotated[dict[str, Any], Output(label="Artifact")]:
    chapters = [ChapterDraft.model_validate(c) for c in chapters]
    body = "\n\n".join(c.markdown for c in chapters) + "\n\n" + resume_text
    return {
        "filename": "miljorapport_borris.docx",
        "size_bytes": len(body),
        "chapter_count": len(chapters),
        "preview": body[:400],
    }


print(f"Registered {len(registry.all())} nodes.")


## 4 · Layer 1 — parallel external fetches via `for-each`

The original `external_data_orchestrator.py` runs N specs through a
`ThreadPoolExecutor`. In Conductor it's a for-each region: pre-build the
deduplicated spec list, hand it to `for-each-start.items` as **static
data** on the node (the canonical way to seed a for-each — `items` uses
the `ConnectionList` widget, which would otherwise wrap a single edge
source into a one-key dict instead of unpacking it). The body fetches
one spec; `for-each-end` collects.

Independent fetches overlap automatically — Conductor schedules each
iteration as soon as it's ready, with sync calls offloaded to
`asyncio.to_thread`. Setting `execution_mode="Parallel"` on the start
node enables thread-pool parallelism inside the region.


In [ ]:
fetch_nodes = [
    GraphNode("start", "for-each-start@1",
                  {"items": unique_lookup_specs(), "execution_mode": "Parallel"}),
    GraphNode("body",  "fetch-ea@1", None),
    GraphNode("end",   "for-each-end@1", None),
    GraphNode("index", "index-results@1", None),
]
fetch_edges = [
    GraphEdge("e2", "start", "body",  "output_1", "spec"),
    GraphEdge("e3", "body",  "end",   "result", "item"),
    GraphEdge("e4", "end",   "index", "result", "results"),
]

t0 = time.perf_counter()
fetch_compiled = compile(
    nodes=fetch_nodes, edges=fetch_edges, registry=registry,
    compound_types=[FOR_EACH],
)
fetch_results = await collect(execute(fetch_compiled))
print(f"Fetched {len(fetch_results['index']['result'])} unique specs "
      f"in {time.perf_counter() - t0:.2f}s")
for k, v in fetch_results["index"]["result"].items():
    print(f"  {k:24s} -> {v.splitlines()[0]}")


## 5 · Layer 2 — per-chapter pipeline with SKIPPED-sentinel routing

Each LLM chapter goes through `generate → evaluate → route → revise/skip
→ ground`. The router emits two parallel branches (revise / pass-through)
using the **SKIPPED sentinel** — the canonical Conductor pattern for
conditional flow when downstream nodes need multiple inputs from the
same branch.

```
  llm-chapter ──┬─> evaluate ──┐
                │              ▼
                └──────────> route-by-score
                              ├─(score < 0.7)──> revise ──┐
                              └─(else)──────────────────────> ground
                                                              ▲
                                              (rev or route.output_3 — only one is live)
```

`route-by-score` returns a 3-tuple: `(draft_for_revise, eval_for_revise,
draft_for_passthrough)`. Two of the three are always `SKIPPED`; the
resolver picks the live one when an input has multiple incoming edges
where all but one are SKIPPED. `revise` is auto-skipped when both its
inputs are SKIPPED.

> Decision nodes + edge guards (`is_decision=True`, `when=` on edges)
> are the right tool when one routed value goes to one downstream node.
> When two routed values must travel together, SKIPPED-sentinel routing
> composes cleanly — that's what we use here.


In [ ]:
chapter_pipeline_nodes = [
    GraphNode("ctx",   "extract-context@1", {"scoping_text": fake_parse_document(b"x")}),
    GraphNode("start", "for-each-start@1",
                  {"items": unique_lookup_specs(), "execution_mode": "Parallel"}),
    GraphNode("body",  "fetch-ea@1", None),
    GraphNode("end",   "for-each-end@1", None),
    GraphNode("index", "index-results@1", None,
              produces={"result": "EA index"}),
    GraphNode("gen",   "llm-chapter@1", {"chapter_id": "ch4", "title": "4. Landskab"},
              consumes={"ea_index": ("index", "result")}),
    GraphNode("eval",  "evaluate@1", None),
    GraphNode("route", "route-by-score@1", None),
    GraphNode("rev",   "revise@1", None),
    GraphNode("grnd",  "ground@1", None),
]
chapter_pipeline_edges = [
    # external fetch fan-out
    GraphEdge("e2", "start", "body",  "output_1", "spec"),
    GraphEdge("e3", "body",  "end",   "result", "item"),
    GraphEdge("e4", "end",   "index", "result", "results"),
    # context → generation → eval → route
    GraphEdge("e5", "ctx",   "gen",   "result",   "project_context"),
    GraphEdge("e6", "gen",   "eval",  "result",   "draft"),
    GraphEdge("e7", "gen",   "route", "result",   "draft"),
    GraphEdge("e8", "eval",  "route", "result",   "eval_result"),
    # SKIPPED-sentinel branches.
    GraphEdge("e9",  "route", "rev",   "output_1", "draft"),
    GraphEdge("e10", "route", "rev",   "output_2", "eval_result"),
    GraphEdge("e11", "route", "grnd",  "output_3", "draft"),  # pass-through
    GraphEdge("e12", "rev",   "grnd",  "result",   "draft"),  # revise path
]

cp_compiled = compile(
    nodes=chapter_pipeline_nodes,
    edges=chapter_pipeline_edges,
    registry=registry,
    compound_types=[FOR_EACH],
)
cp_results = await collect(execute(cp_compiled))

final_draft = ChapterDraft.model_validate(cp_results["grnd"]["result"])
print(f"Final draft: {final_draft.title}  ({len(final_draft.markdown)} chars)")
print(f"Sections: {list(final_draft.sections)}")
print(f"Was revised? {'rev' in cp_results}")


## 6 · Layer 3 — the full report

Bigger flow: 3 template chapters + 4 LLM chapters + 1 cumulative + resume +
docx assembly. Two Conductor primitives keep it tidy:

- **Shared references** broadcast `project_context` and the `EA index` to
  every chapter — no fan-out edge spaghetti.
- The **cumulative chapter** consumes all earlier chapter drafts via
  ConnectionList (one edge per chapter, aggregated into a list).

For brevity we skip the eval / decide / revise / ground sub-pipeline here
— substitute the 5 nodes from §5 inline if you want it on every LLM chapter.


In [ ]:
def build_full_report() -> tuple[list[GraphNode], list[GraphEdge]]:
    nodes: list[GraphNode] = [
        # `parse` produces its scoping text as a shared reference so
        # template chapters can consume it without an explicit edge each.
        GraphNode("parse", "parse-scoping@1", {"blob_id": "demo"},
                  produces={"result": "scoping text"}),
        GraphNode("ctx",   "extract-context@1", None,
                  produces={"result": "project context"}),
        # External data fan-out — items pre-computed from LOOKUP_CONFIG.
        GraphNode("start", "for-each-start@1",
                  {"items": unique_lookup_specs(), "execution_mode": "Parallel"}),
        GraphNode("body",  "fetch-ea@1", None),
        GraphNode("end",   "for-each-end@1", None),
        GraphNode("index", "index-results@1", None,
                  produces={"result": "EA index"}),
    ]
    edges: list[GraphEdge] = [
        GraphEdge("e0", "parse", "ctx",   "result", "scoping_text"),
        GraphEdge("e2", "start", "body",  "output_1", "spec"),
        GraphEdge("e3", "body",  "end",   "result", "item"),
        GraphEdge("e4", "end",   "index", "result", "results"),
    ]

    chapter_node_ids: list[str] = []
    for cid, title in TEMPLATE_CHAPTERS:
        nodes.append(
            GraphNode(cid, "template-chapter@1",
                      {"chapter_id": cid, "title": title},
                      consumes={"scoping_text": ("parse", "result")})
        )
        chapter_node_ids.append(cid)

    # LLM chapters (4–13, condensed to 4). Both project context and EA
    # index are consumed via shared reference — broadcast, no edges.
    for cid, title in LLM_CHAPTERS:
        nodes.append(
            GraphNode(cid, "llm-chapter@1",
                      {"chapter_id": cid, "title": title},
                      consumes={
                          "project_context": ("ctx",   "result"),
                          "ea_index":        ("index", "result"),
                      })
        )
        chapter_node_ids.append(cid)

    # Chapter 14 — cumulative — depends on all earlier chapter drafts.
    cum_id, cum_title = CUMULATIVE_CHAPTER
    nodes.append(GraphNode(cum_id, "cumulative@1", {"title": cum_title}))
    for i, src in enumerate(chapter_node_ids):
        edges.append(GraphEdge(f"cum_{i}", src, cum_id, "result", "earlier_chapters"))
    chapter_node_ids.append(cum_id)

    # Resume — depends on all chapters including ch14.
    nodes.append(GraphNode("resume", "resume@1", None))
    for i, src in enumerate(chapter_node_ids):
        edges.append(GraphEdge(f"res_{i}", src, "resume", "result", "chapters"))

    # Final docx assembly.
    nodes.append(GraphNode("docx", "assemble-docx@1", None))
    for i, src in enumerate(chapter_node_ids):
        edges.append(GraphEdge(f"asm_{i}", src, "docx", "result", "chapters"))
    edges.append(GraphEdge("asm_resume", "resume", "docx", "result", "resume_text"))

    return nodes, edges


fr_nodes, fr_edges = build_full_report()
fr_compiled = compile(
    nodes=fr_nodes, edges=fr_edges, registry=registry, compound_types=[FOR_EACH],
)
print(f"Full report flow: {len(fr_nodes)} nodes, {len(fr_edges)} edges, "
      f"compiled in {len(fr_compiled.execution_order)} steps")
print(f"Type warnings: {len(fr_compiled.type_warnings)}")

t0 = time.perf_counter()
fr_results = await collect(execute(fr_compiled))
print(f"Full report executed in {time.perf_counter() - t0:.2f}s")

artifact = fr_results["docx"]["result"]
print(f"\nArtifact: {artifact['filename']} "
      f"({artifact['chapter_count']} chapters, {artifact['size_bytes']} bytes)")
print("Preview:\n" + artifact["preview"])


## 7 · Streaming execution trace

What did each node actually do? `execute()` is an async generator —
we can listen to its events and build a per-node trace as the flow
runs. The table below shows every node's duration and a snippet of
its output, generated live during a fresh full-report run.


In [ ]:
from IPython.display import Markdown, display


async def trace_run(compiled, *, response=None, checkpoint=None, max_chars=120):
    # Run the flow and return one row per node firing.
    rows: list[dict[str, Any]] = []
    started: dict[str, float] = {}
    if checkpoint is not None:
        evs = resume(compiled, checkpoint, response=response)
    else:
        evs = execute(compiled)
    async for ev in evs:
        et = ev.get("type")
        nid = ev.get("node_id")
        if et == "node_start":
            started[nid] = time.perf_counter()
        elif et in ("node_complete", "node_error", "node_skipped"):
            t0 = started.pop(nid, None)
            ms = int((time.perf_counter() - t0) * 1000) if t0 else 0
            res = ev.get("result")
            snippet = ""
            if et == "node_complete" and isinstance(res, dict):
                v = res.get("result", res)
                if isinstance(v, dict) and "title" in v:
                    snippet = f"[{v.get('chapter_id', '?')}] {v['title']}"
                elif isinstance(v, dict) and "filename" in v:
                    snippet = f"{v['filename']} — {v.get('chapter_count', '?')} kapitler, {v.get('size_bytes', '?')} B"
                elif isinstance(v, dict):
                    snippet = repr(v)
                elif isinstance(v, list):
                    snippet = f"<list len={len(v)}>"
                else:
                    snippet = str(v).replace("\n", " ")
                snippet = snippet[:max_chars] + ("…" if len(snippet) > max_chars else "")
            elif et == "node_error":
                snippet = str(ev.get("error", ""))[:max_chars]
            rows.append({"id": nid, "status": et, "ms": ms, "out": snippet})
        elif et == "flow_paused":
            return rows, ev["checkpoint"]
        elif et in ("flow_error", "flow_cancelled", "flow_timeout"):
            break
    return rows, None


def render_trace(rows, compiled):
    icon = {"node_complete": "✅", "node_error": "❌", "node_skipped": "⏭️"}
    lines = ["| # | node | type | ms | output |", "|---|---|---|---|---|"]
    for i, r in enumerate(rows, 1):
        node = compiled.node_map.get(r["id"])
        ntype = node.type.split("@")[0] if node else r["id"]
        ic = icon.get(r["status"], r["status"])
        out = r["out"].replace("|", "\\|").replace("\n", " ")
        lines.append(f"| {i} | `{r['id']}` | `{ntype}` | {r['ms']} | {ic} {out} |")
    return "\n".join(lines)


# Build a fresh compiled graph so the trace shows a clean run, untainted
# by previous executions.
trace_nodes, trace_edges = build_full_report()
trace_compiled = compile(
    nodes=trace_nodes, edges=trace_edges, registry=registry,
    compound_types=[FOR_EACH],
)

trace_rows, _ = await trace_run(trace_compiled)
print(f"Captured {len(trace_rows)} node firings.")
display(Markdown(render_trace(trace_rows, trace_compiled)))


## 8 · Drill-down — what's actually in each chapter?

Each chapter is built from four named sections. The table above shows
*when* each node fired; this section shows *what* it produced. The
LLM chapters' `existing_conditions` section is the one that consumes
the corresponding EA-Tools result.


In [ ]:
import json as _json

# Re-run end-to-end so we have every node's result without depending on
# stale state from the cells above.
drill_results = await collect(execute(trace_compiled))


def show_chapter(node_id: str, label: str) -> None:
    data = drill_results.get(node_id)
    if not data or "result" not in data:
        print(f"=== {label} ===\n  (no output)\n")
        return
    draft = ChapterDraft.model_validate(data["result"])
    print(f"=== {label} — {draft.title} ===")
    print(f"  ({len(draft.markdown)} tegn, sections: {list(draft.sections)})")
    for section_name, body in draft.sections.items():
        clean = body.strip().replace("\n", " ")
        print(f"  • {section_name:22s} {clean[:160]}{'…' if len(clean) > 160 else ''}")
    print()


# 1) Inputs (what the chapters were given).
print("=== Input — afgrænsningsnotat ===")
print(drill_results["parse"]["result"][:300], "…\n")

print("=== Input — extracted project context ===")
print(drill_results["ctx"]["result"], "\n")

print("=== Input — EA index (4 specs) ===")
ea_index = drill_results["index"]["result"]
for k, v in ea_index.items():
    print(f"--- {k} ---")
    print(v)
    print()

# 2) Each rendered chapter, section by section.
for cid, title in TEMPLATE_CHAPTERS:
    show_chapter(cid, "Template chapter")
for cid, title in LLM_CHAPTERS:
    show_chapter(cid, "LLM chapter")
show_chapter(CUMULATIVE_CHAPTER[0], "Cumulative chapter")

# 3) Resume + final docx artifact.
print("=== Resume (citizen-friendly) ===")
print(drill_results["resume"]["result"], "\n")

print("=== Assembled docx artifact ===")
artifact = drill_results["docx"]["result"]
print(_json.dumps({k: v for k, v in artifact.items() if k != 'preview'},
                  indent=2, ensure_ascii=False))
print("\nFirst 400 chars of body:")
print(artifact["preview"])


## 9 · Visualizing the graph

`conductor.viz.to_mermaid()` renders any `CompiledGraph` as a Mermaid
flowchart. Wrapped in a fenced code block, GitHub renders it natively;
in Jupyter we feed it through `IPython.display.Markdown`. Visual
conventions: decision diamonds, compound subroutines, managed
parallelograms, dashed arrows for shared references.


In [ ]:
from conductor.viz import to_mermaid
from IPython.display import Markdown

mermaid_src = to_mermaid(fr_compiled, direction="LR")
Markdown(f"```mermaid\n{mermaid_src}\n```")


## 10 · Variation — human-in-the-loop dataset-mapping review

In the real system, between the overlap analysis and the full-report run a
human edits the dataset → chapter assignments in the DB. We model that as a
`HumanInputRequired` pause: the flow stops, hands the proposed mapping out
as a checkpoint, and resumes once a reviewer responds.

Same pattern as `06_human_in_the_loop.ipynb`, applied to the envir process.


In [ ]:
@registry.node("propose-mapping", version=1,
               name="Propose mapping",
               description="Suggested dataset → chapter assignments")
def propose_mapping(
    scoping_text: Annotated[str, Textarea(label="Scoping text")],
) -> Annotated[dict[str, list[str]], Output(label="Mapping")]:
    return {
        "ch4":  ["landskab", "beskyttelseslinjer"],
        "ch6":  ["jordforurening"],
        "ch9":  ["vandloeb", "soeer"],
        "ch10": ["natura2000", "habitatkort"],
    }


@registry.node("review-mapping", version=1,
               name="Review mapping",
               description="Pause for human review/edit",
               actor={"kind": "human", "role": "case_officer"})
def review_mapping(
    mapping: Annotated[dict[str, list[str]], Text(label="Proposed mapping")],
) -> Annotated[dict[str, list[str]], Output(label="Approved mapping")]:
    raise HumanInputRequired(
        prompt="Approve or edit the dataset → chapter mapping below.",
        schema={"mapping": "dict[str, list[str]]"},
    )


@registry.node("apply-mapping", version=1,
               name="Apply mapping",
               description="Persists the approved mapping")
def apply_mapping(
    mapping: Annotated[dict[str, list[str]], Text(label="Approved mapping")],
) -> Annotated[str, Output(label="Status")]:
    return f"applied {len(mapping)} chapter assignments"


hitl_nodes = [
    GraphNode("parse", "parse-scoping@1", {"blob_id": "demo"}),
    GraphNode("prop",  "propose-mapping@1", None),
    GraphNode("rev",   "review-mapping@1", None),
    GraphNode("apply", "apply-mapping@1", None),
]
hitl_edges = [
    GraphEdge("e1", "parse", "prop",  "result", "scoping_text"),
    GraphEdge("e2", "prop",  "rev",   "result", "mapping"),
    GraphEdge("e3", "rev",   "apply", "result", "mapping"),
]
hitl_compiled = compile(nodes=hitl_nodes, edges=hitl_edges, registry=registry)

# Phase 1 — run until pause.
import json
try:
    await collect(execute(hitl_compiled))
    print("ERROR: should have paused")
except FlowPausedException as e:
    checkpoint = e.checkpoint
    print(f"Paused at: {checkpoint['waiting_node_id']}")
    print(f"Prompt: {checkpoint['prompt']}")

# Phase 2 — resume with the human's edited mapping.
edited_mapping = {
    "ch4":  ["landskab"],                   # reviewer dropped one
    "ch6":  ["jordforurening", "raastof"],  # reviewer added one
    "ch9":  ["vandloeb", "soeer"],
    "ch10": ["natura2000", "habitatkort"],
}
resumed = await collect(resume(hitl_compiled, json.loads(json.dumps(checkpoint)),
                               response=edited_mapping))
print(f"Resumed: {resumed['apply']['result']}")


## 11 · What we modeled

| Envir module / function | Conductor primitive |
|---|---|
| `pdf_extraction_service.py` | `parse-scoping` node |
| `chapter_generation_service.py:_extract_project_context` | `extract-context` node + `idempotency_key` (matches the envir `HashCache`) |
| `external_data_orchestrator.run_lookups` | `for-each` region: `unique_lookup_specs()` → `fetch-ea` body → `index-results` |
| `lookup_data_config.py:CHAPTER_LOOKUPS` | `LOOKUP_CONFIG` dict consulted by `llm-chapter` |
| `_generate_template_chapter` | `template-chapter` node (renders chapter-specific Danish template prose) |
| `_generate_llm_chapter` | `llm-chapter` node, with `project_context` and `ea_index` pulled in via **shared references** (no edges crossing the parallel chapter wave). Returns four named sections (`legal_basis`, `existing_conditions`, `assessment`, `conclusion`) |
| `revision_service.evaluate_chapter` | `evaluate` node — structural rules + score |
| `revision_service.revise_chapter` | `route-by-score` SKIPPED-sentinel router + `revise` node (revise path runs only when `eval.score < 0.7`) |
| `revision_service.ground_chapter` | `ground` node — appends a *Kilder*-table |
| `_generate_cumulative_chapter` | `cumulative` node, depends on all chapters 1–13 via `ConnectionList`-style edge fan-in |
| `_generate_resume` | `resume` node — citizen-friendly summary |
| `docx_export_service.markdown_to_docx` | `assemble-docx` node |
| Dataset → chapter mapping review (DB step) | `review-mapping` node raising `HumanInputRequired` |
| `agent_service` LLM split-completion fallback | per-node `max_retries=1` + a manual draft-fallback node would model it; not exercised here |

Things deliberately out of scope of the example so it stays readable:

- The 7 parallel LLM judges inside `evaluate` (would be a sub-flow of its own).
- Map embedding (`map_embedding.replace_map_markers`) — pure transform; one
  more node, nothing structurally new.
- The `revise-stream` POST endpoint — same shape as the HITL section above.
